# LangGraph — build the agent loop as a graph

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhelamay/CPUT_ICAILY_AI2026_MaganLLM_WORKSHOP/blob/main/SA_LIB_WORKSHOP_Day2/01_langgraph/01_langgraph.ipynb)

**Author:** Dr. Nasser Mooman · Magan AI Inc. · nmooman@gmail.com · written with help from Claude (Anthropic)

> ⚠️ **Teaching code — do not use in production.** Short on purpose: no authentication, input validation, prompt-injection protection, error handling or tests. LLM answers can be wrong. Provided as is, without warranty.

**SA-LIB AI Workshop 2026 — Day 2: AI Agents**

LangGraph is the engine under LangChain's `create_agent`. Here we build the graph **by hand** (state + nodes + edges), then go further: a multi-agent loop and a deep-dive with a router, human approval (`interrupt`) and streaming. See `docs/langgraph_explained.html`.

Same example in every framework: **(1) first agent → (2) tools → (3) memory → (4) fallback → (5) multi-agent → complete agent**.

## 0. Setup
Install the packages (takes ~1 minute in Colab).

In [ ]:
%pip install -q "langgraph>=1.0" "langchain-core>=1.0" "langchain-openai>=1.0" "python-dotenv" "ddgs"

### API keys
**Bring your own LLM API key** (Groq and Gemini have free tiers). No key is needed only if you already run a local server (Ollama, SGLang or vLLM): set `LLM_PROVIDER=ollama`, `OLLAMA_BASE_URL` and `LLM_MODEL`.

* **Colab:** click the 🔑 *Secrets* icon on the left, add e.g. `GROQ_API_KEY`, and enable notebook access.
* **VS Code / Jupyter / Linux:** put your keys in the `.env` file in the repo root (copy `.env.example`).
* Otherwise you'll be asked to paste a key below.

Choose the model provider with `LLM_PROVIDER`: `groq` · `gemini` · `openai` · `deepseek` · `openrouter` · `ollama`.

In [ ]:
import os, getpass
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))          # VS Code / Linux: read ../.env
try:                                           # Colab: read the Secrets panel
    from google.colab import userdata
    for key in ['GROQ_API_KEY', 'GOOGLE_API_KEY', 'OPENAI_API_KEY', 'DEEPSEEK_API_KEY', 'OPENROUTER_API_KEY', 'LANGSMITH_API_KEY', 'LANGFUSE_PUBLIC_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_HOST', 'LLM_PROVIDER', 'LLM_MODEL']:
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except ImportError:
    pass

os.environ.setdefault("LLM_PROVIDER", "groq")   # <- change the provider here if you like
provider = os.environ["LLM_PROVIDER"]
key_name = {"groq": "GROQ_API_KEY", "gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY",
            "deepseek": "DEEPSEEK_API_KEY", "openrouter": "OPENROUTER_API_KEY"}.get(provider)
if key_name and not os.getenv(key_name):
    os.environ[key_name] = getpass.getpass(f"{key_name}: ")
print("provider:", provider)

### Shared helper: `workshop_common.py`
Reads the provider settings, and holds the two tools every example uses: `web_search` (free DuckDuckGo) and `word_count`.

In [ ]:
%%writefile workshop_common.py
"""
workshop_common.py — shared helpers for every framework folder in this workshop.

(the SAME file is copied into each folder so every folder works on its own —
 edit shared/workshop_common.py and run `python tools/sync_common.py` to update the copies)

What it gives you:
  get_llm_config()   -> which provider / model / base_url / api key to use (read from .env)
  fallback_order()   -> the list of providers to try, in order, for the fallback examples
  ping()             -> quick "are you alive?" check against a provider
  web_search()       -> free DuckDuckGo search (no key)
  word_count()       -> the tiny "tool you wrote yourself" from the original guide

Every provider below speaks the OpenAI chat-completions protocol, which is why every
framework in this workshop can use every provider: we just change base_url + api_key + model.
"""

from __future__ import annotations

import os
from dataclasses import dataclass

from dotenv import find_dotenv, load_dotenv

# walks up from where you run python, so the .env in the repo root is always found
load_dotenv(find_dotenv(usecwd=True))

PROVIDERS = {
    #  name        base_url                                                     api-key env var       default model
    "groq":       ("https://api.groq.com/openai/v1",                           "GROQ_API_KEY",       "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY",     "gemini-2.5-flash"),
    "openai":     ("https://api.openai.com/v1",                                "OPENAI_API_KEY",     "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com",                                 "DEEPSEEK_API_KEY",   "deepseek-chat"),
    "openrouter": ("https://openrouter.ai/api/v1",                             "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),  None,                 "qwen2.5:7b"),
}


@dataclass
class LLMConfig:
    provider: str
    model: str
    base_url: str
    api_key: str

    @property
    def litellm_model(self) -> str:
        """Model name in LiteLLM format (used by CrewAI and Google ADK): 'openai/<model>'
        means 'an OpenAI-compatible endpoint' — combined with base_url it works for every provider."""
        return f"openai/{self.model}"


def get_llm_config(provider: str | None = None) -> LLMConfig:
    """Read LLM_PROVIDER / LLM_MODEL / <PROVIDER>_API_KEY from the environment."""
    provider = (provider or os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    if provider not in PROVIDERS:
        raise ValueError(f"Unknown LLM_PROVIDER '{provider}'. Choose one of: {', '.join(PROVIDERS)}")
    base_url, key_env, default_model = PROVIDERS[provider]
    api_key = os.getenv(key_env) if key_env else "ollama"  # ollama runs locally, no key
    if not api_key:
        raise RuntimeError(f"{key_env} is missing — add it to your .env file (see .env.example).")
    # LLM_MODEL only overrides the model of the MAIN provider, not of the fallbacks
    main_provider = (os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    model = (os.getenv("LLM_MODEL") if provider == main_provider else None) or default_model
    return LLMConfig(provider, model, base_url, api_key)


def fallback_order() -> list[str]:
    """Providers to try, in order. Default: your LLM_PROVIDER first, then LLM_FALLBACKS (groq,gemini)."""
    main = (os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    extra = [p.strip().lower() for p in os.getenv("LLM_FALLBACKS", "groq,gemini").split(",") if p.strip()]
    return list(dict.fromkeys([main, *extra]))  # remove duplicates, keep order


def ping(cfg: LLMConfig) -> None:
    """Send a 1-token request. Raises an exception if the provider is down, rate-limited or the key is wrong."""
    from openai import OpenAI

    OpenAI(base_url=cfg.base_url, api_key=cfg.api_key, timeout=20, max_retries=0).chat.completions.create(
        model=cfg.model, messages=[{"role": "user", "content": "ping"}], max_tokens=1
    )


def web_search(query: str) -> str:
    """Search the web with DuckDuckGo (free, no key) and return the top results as text."""
    try:
        from ddgs import DDGS

        results = DDGS().text(query, max_results=3)
        return "\n".join(f"- {r['title']}: {r['body']} ({r['href']})" for r in results) or "no results"
    except Exception as e:  # the free search is rate-limited: return the error instead of crashing
        return f"web search failed ({e}). try again in a few seconds."


def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())

## 1. `01_first_agent.py`

LangGraph 01 — your first agent
same task as the original guide, but we build the graph OURSELVES instead of calling create_agent.
a LangGraph app is:  STATE (the data)  +  NODES (python functions)  +  EDGES (who runs next)

    START --> [chatbot] --> END

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, MessagesState, StateGraph

from workshop_common import get_llm_config

# the brain: any provider from your .env (groq by default) — they are all OpenAI-compatible
cfg = get_llm_config()
model = ChatOpenAI(model=cfg.model, base_url=cfg.base_url, api_key=cfg.api_key, temperature=0)

SYSTEM = {"role": "system", "content": "You are a helpful assistant. Be concise and accurate."}


# a NODE is a function: it receives the current state and returns an UPDATE to the state
def chatbot(state: MessagesState):
    reply = model.invoke([SYSTEM] + state["messages"])
    return {"messages": [reply]}  # MessagesState APPENDS new messages (it doesn't overwrite)


# wire the graph
graph = StateGraph(MessagesState)
graph.add_node("chatbot", chatbot)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)
agent = graph.compile()

result = agent.invoke({"messages": [{"role": "user", "content": "explain what an AI agent is in two sentences."}]})
print(f"[{cfg.provider}:{cfg.model}]", result["messages"][-1].content)

## 2. `02_agent_with_tools.py`

LangGraph 02 — giving your agent tools
the agent LOOP as a graph. this is exactly what create_agent builds for you:

    START --> [agent] --tool call?--> [tools] --+
                 ^                               |
                 +-------------------------------+
              [agent] --no tool call--> END

In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

import workshop_common as wc

cfg = wc.get_llm_config()
model = ChatOpenAI(model=cfg.model, base_url=cfg.base_url, api_key=cfg.api_key, temperature=0)


# tool 1 — web search (free, no key)
@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information."""
    return wc.web_search(query)


# tool 2 — a function you wrote yourself. the DOCSTRING is what the model reads.
@tool
def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())


tools = [web_search, word_count]
model_with_tools = model.bind_tools(tools)  # tells the model which tools exist
SYSTEM = {"role": "system", "content": "You are a helpful assistant. Use your tools when they help answer accurately."}


def agent_node(state: MessagesState):          # PLAN: the model decides — answer, or call tools?
    return {"messages": [model_with_tools.invoke([SYSTEM] + state["messages"])]}


graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))        # ACT: runs the tools the model asked for
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition)  # tool calls? -> "tools"  else -> END
graph.add_edge("tools", "agent")                # OBSERVE: tool results go back to the model
agent = graph.compile()

result = agent.invoke({"messages": [{"role": "user", "content":
    "search for the latest langchain version, then tell me how many words your answer is."}]})

# print every step so you can SEE the loop
for msg in result["messages"]:
    calls = getattr(msg, "tool_calls", None)
    print(f"[{msg.type}]", calls if calls else str(msg.content)[:300])

## 3. `03_agent_with_memory.py`

LangGraph 03 — giving your agent memory
compile the graph with a CHECKPOINTER. after every step LangGraph saves the state.
a thread_id picks which saved conversation to continue.

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph

from workshop_common import get_llm_config

cfg = get_llm_config()
model = ChatOpenAI(model=cfg.model, base_url=cfg.base_url, api_key=cfg.api_key, temperature=0)
SYSTEM = {"role": "system", "content": "You are a helpful assistant. Be concise and accurate."}


def chatbot(state: MessagesState):
    return {"messages": [model.invoke([SYSTEM] + state["messages"])]}


graph = StateGraph(MessagesState)
graph.add_node("chatbot", chatbot)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)
agent = graph.compile(checkpointer=InMemorySaver())   # <- the only change: memory

config = {"configurable": {"thread_id": "chat-1"}}      # same id = same conversation

r1 = agent.invoke({"messages": [{"role": "user", "content": "hi! my name is m0h."}]}, config)
print(r1["messages"][-1].content)

r2 = agent.invoke({"messages": [{"role": "user", "content": "what's my name?"}]}, config)
print(r2["messages"][-1].content)

# a different thread_id = a fresh conversation with no memory
r3 = agent.invoke({"messages": [{"role": "user", "content": "what's my name?"}]},
                  {"configurable": {"thread_id": "chat-2"}})
print("[chat-2]", r3["messages"][-1].content)

# bonus: you can read the saved state of any thread
print("messages saved in chat-1:", len(agent.get_state(config).values["messages"]))

## 4. `04_agent_with_fallback.py`

LangGraph 04 — the fallback setup
try each provider in order (LLM_PROVIDER, then LLM_FALLBACKS in .env). first one that answers wins.

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, MessagesState, StateGraph

from workshop_common import fallback_order, get_llm_config, ping


def get_model():
    for provider in fallback_order():           # e.g. ["groq", "gemini"]
        try:
            cfg = get_llm_config(provider)
            ping(cfg)                           # quick test that it actually responds
            print(f"using {provider} ({cfg.model})")
            return ChatOpenAI(model=cfg.model, base_url=cfg.base_url, api_key=cfg.api_key, temperature=0)
        except Exception as e:
            print(f"{provider} failed ({str(e)[:80]}); trying the next one")
    raise RuntimeError("every provider failed — check your keys in .env")


model = get_model()


def chatbot(state: MessagesState):
    return {"messages": [model.invoke(state["messages"])]}


graph = StateGraph(MessagesState)
graph.add_node("chatbot", chatbot)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)
agent = graph.compile()

result = agent.invoke({"messages": [{"role": "user", "content": "say hi and tell me which model you are."}]})
print(result["messages"][-1].content)

## 5. `05_multi_agent.py`

LangGraph 05 — multi-agent: researcher -> writer -> reviewer (with a feedback loop)
each "agent" is a node. they share one STATE object (a typed dict) instead of just messages.

    START -> [researcher] -> [writer] -> [reviewer] --too long?--> [writer]  (loop, max 3 drafts)
                                                    --ok---------> END

In [ ]:
from typing import TypedDict

from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

from workshop_common import get_llm_config, web_search, word_count

cfg = get_llm_config()
model = ChatOpenAI(model=cfg.model, base_url=cfg.base_url, api_key=cfg.api_key, temperature=0)
MAX_WORDS = 60


class TeamState(TypedDict, total=False):   # the shared whiteboard all agents read & write
    topic: str
    notes: str
    draft: str
    words: int
    attempts: int


def researcher(state: TeamState):
    results = web_search(state["topic"])
    notes = model.invoke(f"Summarise these search results as 5 short bullet facts:\n{results}").content
    print(f"[researcher] collected notes ({word_count(notes)} words)")
    return {"notes": notes}


def writer(state: TeamState):
    feedback = ""
    if state.get("draft"):
        feedback = f"\nYour previous draft had {state['words']} words — too long. Make it shorter."
    draft = model.invoke(
        f"Write ONE paragraph of at most {MAX_WORDS} words about '{state['topic']}' "
        f"for workshop attendees, using these notes:\n{state['notes']}{feedback}"
    ).content
    attempts = state.get("attempts", 0) + 1
    print(f"[writer] draft #{attempts}")
    return {"draft": draft, "attempts": attempts}


def reviewer(state: TeamState):
    words = word_count(state["draft"])     # plain python can be an "agent" step too
    print(f"[reviewer] {words} words")
    return {"words": words}


def route_after_review(state: TeamState) -> str:
    if state["words"] > MAX_WORDS and state["attempts"] < 3:
        return "writer"                     # send it back
    return END


graph = StateGraph(TeamState)
graph.add_node("researcher", researcher)
graph.add_node("writer", writer)
graph.add_node("reviewer", reviewer)
graph.add_edge(START, "researcher")
graph.add_edge("researcher", "writer")
graph.add_edge("writer", "reviewer")
graph.add_conditional_edges("reviewer", route_after_review, ["writer", END])
team = graph.compile()

print(team.get_graph().draw_mermaid())      # paste into https://mermaid.live to see the picture
final = team.invoke({"topic": "the latest langchain version"})
print("\nFINAL:\n" + final["draft"])

## 6. `agent.py`

LangGraph — the complete agent (fallback + tools + memory), graph built by hand

In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

import workshop_common as wc


# --- fallback: try each provider until one answers ---
def get_model():
    for provider in wc.fallback_order():
        try:
            cfg = wc.get_llm_config(provider)
            wc.ping(cfg)
            print(f"using {provider} ({cfg.model})")
            return ChatOpenAI(model=cfg.model, base_url=cfg.base_url, api_key=cfg.api_key, temperature=0)
        except Exception as e:
            print(f"{provider} failed ({str(e)[:80]}); trying the next one")
    raise RuntimeError("every provider failed — check your keys in .env")


# --- tools ---
@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information."""
    return wc.web_search(query)


@tool
def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())


tools = [web_search, word_count]
model = get_model().bind_tools(tools)
SYSTEM = {"role": "system", "content": "You are a helpful assistant. Use your tools when they help answer "
                                       "accurately, and remember the conversation."}


def agent_node(state: MessagesState):
    return {"messages": [model.invoke([SYSTEM] + state["messages"])]}


# --- graph + memory ---
graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition)
graph.add_edge("tools", "agent")
agent = graph.compile(checkpointer=InMemorySaver())


def main():
    config = {"configurable": {"thread_id": "chat-1"}}
    r1 = agent.invoke({"messages": [{"role": "user", "content": "my name is m0h. search for the latest "
                      "langchain version, then tell me how many words your answer is."}]}, config)
    print(r1["messages"][-1].content)
    r2 = agent.invoke({"messages": [{"role": "user", "content": "what's my name?"}]}, config)
    print(r2["messages"][-1].content)


if __name__ == "__main__":
    main()

## 7. `06_langgraph_deep_dive.py`

LangGraph 06 — deep dive: a research assistant with a router, a tool loop and human approval
(explained step by step in docs/langgraph_explained.html)

                      +--> [chat] -----------------------------+
  START --> [router] -+                                        +--> [human_review] --approved--> [publish] --> END
                      +--> [researcher] <--> [tools]  ---------+            |
                                                                            +--rejected--> [researcher] (try again)

what this shows, in one file:
  1. a CUSTOM STATE with several fields (not just messages) and a reducer
  2. a ROUTER node that uses structured output to pick a branch (conditional edges)
  3. a TOOL-CALLING LOOP (researcher <-> tools)
  4. HUMAN-IN-THE-LOOP with interrupt(): the graph pauses and waits for your approval
  5. a CHECKPOINTER: the paused graph is saved, and resumed later with Command(resume=...)
  6. STREAMING: we print each node's update as it happens

     (set AUTO_APPROVE=1 to skip the keyboard prompt, e.g. in CI)

In [ ]:
import os
from typing import Annotated, Literal, TypedDict

from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt
from pydantic import BaseModel, Field

import workshop_common as wc

cfg = wc.get_llm_config()
llm = ChatOpenAI(model=cfg.model, base_url=cfg.base_url, api_key=cfg.api_key, temperature=0)


# ---------------------------------------------------------------- 1. STATE
class State(TypedDict, total=False):
    question: str
    route: str                                            # "chat" or "research"
    messages: Annotated[list[AnyMessage], add_messages]   # reducer: APPEND, don't overwrite
    draft: str
    feedback: str                                         # what the human said when rejecting


# ---------------------------------------------------------------- tools
@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information."""
    return wc.web_search(query)


@tool
def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())


TOOLS = [web_search, word_count]


# ---------------------------------------------------------------- 2. ROUTER (structured output)
class Route(BaseModel):
    route: Literal["chat", "research"] = Field(
        description="'research' if answering needs current / factual info from the web, otherwise 'chat'")


def router(state: State):
    decision = llm.with_structured_output(Route, method="function_calling").invoke(
        f"Classify this question: {state['question']}")
    return {"route": decision.route}


def pick_branch(state: State) -> str:          # a conditional edge is just a function returning a node name
    return state["route"]


# ---------------------------------------------------------------- branch A: plain chat
def chat(state: State):
    return {"draft": llm.invoke(state["question"]).content}


# ---------------------------------------------------------------- branch B: 3. tool-calling loop
RESEARCHER_PROMPT = SystemMessage("You are a careful researcher. Use web_search for facts. Answer in under 80 words.")


def researcher(state: State):
    new = []
    if not state.get("messages"):     # first visit: start the conversation with the question
        new.append(HumanMessage(state["question"]))
    if state.get("feedback"):         # coming back after the human rejected the draft
        new.append(HumanMessage(f"A reviewer rejected your answer: {state['feedback']}. Improve it."))
    history = (state.get("messages") or []) + new
    reply = llm.bind_tools(TOOLS).invoke([RESEARCHER_PROMPT] + history)
    update = {"messages": new + [reply], "feedback": ""}
    if not reply.tool_calls:          # no more tools needed -> this answer is the draft
        update["draft"] = reply.content
    return update


def after_researcher(state: State) -> str:
    return "tools" if tools_condition(state) == "tools" else "human_review"


# ---------------------------------------------------------------- 4. HUMAN-IN-THE-LOOP
def human_review(state: State):
    # interrupt() PAUSES the graph here and returns control to the caller.
    # whatever the caller later sends with Command(resume=...) becomes `decision`.
    decision = interrupt({"draft": state["draft"], "ask": "approve? (yes / or type feedback)"})
    if str(decision).strip().lower() in {"y", "yes", "ok", "approve"}:
        return Command(goto="publish")
    if state["route"] == "chat":
        return Command(goto="chat")
    return Command(goto="researcher", update={"feedback": str(decision)})


def publish(state: State):
    print(f"\n PUBLISHED ({wc.word_count(state['draft'])} words):\n{state['draft']}")
    return {}


# ---------------------------------------------------------------- build the graph
g = StateGraph(State)
g.add_node("router", router)
g.add_node("chat", chat)
g.add_node("researcher", researcher)
g.add_node("tools", ToolNode(TOOLS))
g.add_node("human_review", human_review, destinations=("publish", "researcher", "chat"))
g.add_node("publish", publish)

g.add_edge(START, "router")
g.add_conditional_edges("router", pick_branch, {"chat": "chat", "research": "researcher"})
g.add_edge("chat", "human_review")
g.add_conditional_edges("researcher", after_researcher, ["tools", "human_review"])
g.add_edge("tools", "researcher")
g.add_edge("publish", END)

app = g.compile(checkpointer=InMemorySaver())  # 5. checkpointer = pause / resume / memory


# ---------------------------------------------------------------- 6. run with streaming
def run(question: str, thread_id: str):
    config = {"configurable": {"thread_id": thread_id}}
    payload = {"question": question}
    while True:
        for update in app.stream(payload, config, stream_mode="updates"):
            for node, change in update.items():
                if node == "__interrupt__":
                    continue
                keys = ", ".join(change.keys()) if isinstance(change, dict) else ""
                print(f"  -> {node:<13} updated: {keys}")
        state = app.get_state(config)
        if not state.next:                     # nothing left to run -> finished
            return
        draft = state.values.get("draft", "")   # we are paused in human_review
        print(f"\n--- DRAFT for review ---\n{draft}\n")
        answer = "yes" if os.getenv("AUTO_APPROVE") else input("approve? (yes / or type feedback): ")
        payload = Command(resume=answer)       # resume the paused graph with the human's answer


if __name__ == "__main__":
    print(app.get_graph().draw_mermaid())
    print("\n=== question 1 (should route to CHAT) ===")
    run("Give me a friendly one-sentence welcome for workshop attendees in Libya.", "t1")
    print("\n=== question 2 (should route to RESEARCH) ===")
    run("What is the latest stable version of LangGraph, and when was it released?", "t2")

---
*SA-LIB AI Workshop 2026 · based on [build-ai-agents-free](https://github.com/Moh4696/build-ai-agents-free) by m0h (MIT)*